<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 15px; height: 80px"> 
# Week 4, Lab 1: Classificationn & Penguins
<br>
<br>

> _Author:_ Tim Book

In [1]:
# Import everything here!
# As we import more and more things in our course, it will become crucial that
# our imports are all in one place. Put them all in this cell. Do not write any
# import statements below this cell!

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

# Part I: Penguins!

We'll start out building some simple logistic regressions on the Palm Penguin dataset. You can read more about it [here](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data?resource=download).

![](assets/penguins.png)

### 1. Read the data in
Check out the `.head()` if you like.

In [2]:
pen = pd.read_csv('data/penguins.csv')
pen.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


### 2. Fix the odd `sex` value
Look at the different values contained in the `sex` column. Replace the value that does not belong with a missing value.

In [3]:
pen['sex'].value_counts()

MALE      168
FEMALE    165
.           1
Name: sex, dtype: int64

In [4]:
pen.loc[pen['sex'] == '.', 'sex'] = np.nan

### 3. Display all rows in which any observation is missing

In [5]:
pen[pen.isna().any(axis=1)]

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN
246,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN
286,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN
324,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN
336,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN


### 4. Drop the rows in which all of the numeric columns are missing.
Make sure to keep the 9 rows where _only_ `sex` is missing!

In [6]:
num_cols = pen.select_dtypes('number').columns
pen.dropna(subset=num_cols, inplace=True)

### 5. Repeat Step 3 to confirm that Step 4 worked!

You may be certain in your code, but it's _always_ important to make sure your code worked as intended! Show all the rows with missing values again and visually confirm you dropped only the intended rows.

In [7]:
pen[pen.isna().any(axis=1)]

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN
246,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN
286,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN
324,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN
336,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN


### 6.  Classification Model: Imputation!

There are now 9 rows that have a missing value for `sex`, and we don't want to drop those perfectly good rows. We could use mode-imputation and replace this NAs with the most common `sex` in our dataset - but that's too easy! Instead, let's fill in those values **by predicting them with logistic regression!**

#### 6a:
Create a matrix `X` consisting of the 4 numeric columns in our dataset. Also create a `y` that's just the `sex` column. Binarize `y` into 0s and 1s.

In [8]:
pen.sex.value_counts()

MALE      168
FEMALE    165
Name: sex, dtype: int64

In [9]:
X = pen[[
    'culmen_length_mm',
    'culmen_depth_mm',
    'flipper_length_mm',
    'body_mass_g'
]]

y = pen['sex'].map({'MALE': 1, 'FEMALE': 0})

#### 6b:
Divide `X` and `y` into training and testing data based on whether or not `sex` is present. That is:
* `X_train` should be all rows in which `y` **is not** NA
* `X_test` should be all rows in which `y` **is** NA
* `y_train` should be all the non-NA values of `y`
* `y_test` does not exist! Our goal is to predict the missing `sex` values

In [10]:
X_train = X[y.notna()]
X_test = X[y.isna()]
y_train = y[y.notna()]

#### 6c:
Fit a logistic regression model that can predict penguin `sex` using the dataframes you've created. This model doesn't have to be good! Just has to be ok. You can check out the training score to check on how it's performing.

In [11]:
lr_impute = LogisticRegression(penalty='none')
lr_impute.fit(X_train, y_train)

LogisticRegression(penalty='none')

In [12]:
lr_impute.score(X_train, y_train)

0.8318318318318318

#### 6d:
Create predictions for the 9 rows with missing `sex`. Recode the 0s and 1s back into `MALE` and `FEMALE`.

In [13]:
y_pred = lr_impute.predict(X_test)
y_pred = np.where(y_pred == 1, 'MALE', 'FEMALE')

#### 6e:

Create a copy of your penguin data (use `.copy()`) and replace the missing values with the ones you just created. Replacing subsets of dataframes like this is tricky, so I've provided an example on how to do this with `.loc[]`:

In [14]:
# SAMPLE CODE ON REPLACING VALUES IN A DATAFRAME
df = pd.DataFrame({
    'c1': ['a', 'a', 'b', 'a', 'b'],
    'c2': [1, np.nan, 5, 4, np.nan]
})

replacement_data = [100, 200]

print(df)
print('='*30)

# .loc[] allows us to select exactly the subset of the dataframe
# with which to replace
df.loc[df['c2'].isna(), 'c2'] = replacement_data

print(df)

  c1   c2
0  a  1.0
1  a  NaN
2  b  5.0
3  a  4.0
4  b  NaN
  c1     c2
0  a    1.0
1  a  100.0
2  b    5.0
3  a    4.0
4  b  200.0


In [15]:
pen_imputed = pen.copy()
pen_imputed.loc[pen_imputed['sex'].isna(), 'sex'] = y_pred

#### 6f:

Verify! Confirm that there are now no more missing values in your new data set.

In [16]:
pen_imputed.isna().sum()

species              0
island               0
culmen_length_mm     0
culmen_depth_mm      0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64

### 7: Train-Test Split
Create our `X` and `y` matrices and carry out a train-test split on our new penguin data. The default split size is fine. Make sure to use a seed. It doesn't matter _what_ seed you use, as long as you do use one!

We're going to be predicting penguin species. Use every other provided variable as an x-variable **except island! Do NOT use island!**. You will have to transform variables yourself where appropriate!

In [17]:
# X = pd.get_dummies(columns=['island'], data=pen_imputed.drop(columns=['species']), drop_first=True)

X = pen_imputed.drop(columns=['species', 'island'])
X['sex'] = X['sex'].map({'MALE': 1, 'FEMALE': 0})

y = pen_imputed['species']

In [18]:
X.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.1,18.7,181.0,3750.0,1
1,39.5,17.4,186.0,3800.0,0
2,40.3,18.0,195.0,3250.0,0
4,36.7,19.3,193.0,3450.0,0
5,39.3,20.6,190.0,3650.0,1


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1234)

### 8: Classification Model: Prediction!

Use the data you just created to carry out logistic regression. (Yes, you will get a convergence warning).

In [20]:
lr_sex = LogisticRegression(penalty='none')
lr_sex.fit(X_train, y_train)

/home/tim/.local/lib/python3.6/site-packages/sklearn/linear_model/_logistic.py:764: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG)


LogisticRegression(penalty='none')

### 9: Evaluate

What is the training score? Test score? Create a confusion matrix for your species predictions.

In [21]:
lr_sex.score(X_train, y_train)

0.98828125

In [22]:
lr_sex.score(X_test, y_test)

1.0

In [23]:
y_test

65        Adelie
311       Gentoo
264       Gentoo
98        Adelie
38        Adelie
         ...    
230       Gentoo
336       Gentoo
24        Adelie
170    Chinstrap
56        Adelie
Name: species, Length: 86, dtype: object

In [24]:
metrics.confusion_matrix(y_test, lr_sex.predict(X_test))

array([[40,  0,  0],
       [ 0, 20,  0],
       [ 0,  0, 26]])

### 10: Discuss!
How is this model? Good or bad?

**Solution:** This model is very good! In fact, it is perfect even on the testing data. This is tutorial-quality data, so this isn't too surprising.

# Part II: Malware Prediction

Next, we'll use a data consisting of 241 procedurally generated binary features to determine whether or not software is malware or not. Since our feature aren't human interpretable, and there are so many of them, manually picking and choosing them is infeasible.

![](assets/malware.png)

### 1. Read in the data

We'll be using the TUANDROMD data. You can read more about it [here](https://archive.ics.uci.edu/dataset/813/tunadromd).

Additionally, there is a single row that is entirely missing. Go ahead and drop it.

In [25]:
tuan = pd.read_csv('data/TUANDROMD.csv').dropna()
tuan.head()

,ACCESS_ALL_DOWNLOADS,ACCESS_CACHE_FILESYSTEM,ACCESS_CHECKIN_PROPERTIES,ACCESS_COARSE_LOCATION,ACCESS_COARSE_UPDATES,ACCESS_FINE_LOCATION,ACCESS_LOCATION_EXTRA_COMMANDS,ACCESS_MOCK_LOCATION,ACCESS_MTK_MMHW,ACCESS_NETWORK_STATE,...,Landroid/telephony/TelephonyManager;->getLine1Number,Landroid/telephony/TelephonyManager;->getNetworkOperator,Landroid/telephony/TelephonyManager;->getNetworkOperatorName,Landroid/telephony/TelephonyManager;->getNetworkCountryIso,Landroid/telephony/TelephonyManager;->getSimOperator,Landroid/telephony/TelephonyManager;->getSimOperatorName,Landroid/telephony/TelephonyManager;->getSimCountryIso,Landroid/telephony/TelephonyManager;->getSimSerialNumber,Lorg/apache/http/impl/client/DefaultHttpClient;->execute,Label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### 2. Separate the Data into X and y

For your `X`, you'll use everything except the label. Your `y` will be the column called `Label`. Currently, the label indicates goodware. **_RECODE THE LABEL TO BE THE OPPOSITE!_** We want to detect malware, so we want 1 = Malware and 0 = Goodware.

In [26]:
X = tuan.drop(columns=['Label'])
y = 1 - tuan['Label']

### 3a. Observe the class (im)balance

What proportion of our data are malware?

In [27]:
# 20.14% of our data are malware - the rest are Goodware
1 - y.mean()

0.7986111111111112

### 3b. What does this mean?

We have some class imbalance. What will this mean when looking at our model's accuracy?

In [28]:
# It means we'll need to be a baseline accuracy of 79.9%. Anything
# lower than this score is worse than random guessing.

### 3c. Error Types

Suppose that you plan on using a piece of software, and you'll use this model to determine whether or not it's malware. In this context, explain what a Type I Error is and what a Type II Error is. Which is worse?

In [29]:
# Type I error = Determining malware when in fact it's safe
# Type II error = Determining goodware when in fact it's malicious

# A type II error here is much worse - booting up malicious software by accident
# can have catastrophic results.

### 4. Train Test Split

Default split size is ok. Use a random state of `7777`.

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=7777)

### 5a. Logistic Regression

Fit an _unregularized_ logistic regression to this data.

_Note:_ In this model, and in many subsequent models, you'll get tons of convergence errors. It's generally a bad idea to ignore them by using the `warnings` library. Instead, to get rid of them, you can increase the `max_iter` argument to logistic regression. We'll learn more about this during our gradient descent lesson in a few weeks.

In [31]:
m1 = LogisticRegression(penalty='none', max_iter=1000)
m1.fit(X_train, y_train)

LogisticRegression(max_iter=1000, penalty='none')

### 5b. Scoring

Show the train and testing score for this model. In words, how does it perform?

In [32]:
print(m1.score(X_train, y_train))
print(m1.score(X_test, y_test))

0.996415770609319
0.982078853046595


In [33]:
# The model fits very well! It beats the baseline significantly.

### 5c. Confusion Matrix

Show a confusion matrix on the testing data. How many Type II errors do we commit? What is the **recall** for our results? Explain the recall in words.

In [34]:
metrics.confusion_matrix(y_test, m1.predict(X_test))

array([[893,   5],
       [ 15, 203]])

In [35]:
# We've got 15 Type II error in our results!
# The recall is 203 / 218 = 93.2%
# That is, we correctly detected 93.2% of all malicious software.

In [36]:
203/218

0.9311926605504587

### 6. Regularized Logistic Regression

Repeat all of Step 5, except this time use an $l_2$ penalty on your logistic regression. Use built-in cross validation to optimize the best tuning parameter for regularization. Make sure to evaluate your model, show a confusion matrix, and interpret the recall as well!

_Note:_ Yes, it is required for us to standard scale our data for regularized problems. However, if you look through the data, you'll notice all of our variables are binary - that is - they are already all on a common scale an that scale is meaningful! So **let's not scale this dataset today**. This is an exception to our rule.

_Note on wording:_ We learned about **ridge regression** for regression problems. However, we're doing classification here! Ridge regression is specifically for regression. When we apply an $l_2$ penalty onto logistic regression, that's all it is: "logistic regression with an $l_2$ penalty" or sometimes "logistic regression with a ridge-like penalty". It is _NOT_ "ridge regression"!

In [37]:
%%time
m2 = LogisticRegressionCV(penalty='l2', Cs=100, cv=5, max_iter=1000)
m2.fit(X_train, y_train)

CPU times: user 1min 53s, sys: 4min 9s, total: 6min 2s
Wall time: 23 s


LogisticRegressionCV(Cs=100, cv=5, max_iter=1000)

In [38]:
print(m2.score(X_train, y_train))
print(m2.score(X_test, y_test))

0.9946236559139785
0.989247311827957


In [39]:
m2.C_

array([31.2571585])

In [40]:
metrics.confusion_matrix(y_test, m2.predict(X_test))

array([[892,   6],
       [  6, 212]])

In [41]:
# Now we've got 6 Type II error in our results!
# The recall is 212 / 218 = 97.2%
# That is, we correctly detected 97.2% of all malicious software.

In [42]:
212/218

0.9724770642201835

### 7. Again... but this time for $l_1$!

Just like for Step 6, carry out all of Step 5 but this time for $l_1$ regularization.

In [43]:
%%time
m3 = LogisticRegressionCV(penalty='l1', Cs=100, cv=5, solver='liblinear', max_iter=1000)
m3.fit(X_train, y_train)

CPU times: user 10 s, sys: 7.76 s, total: 17.8 s
Wall time: 8.33 s


LogisticRegressionCV(Cs=100, cv=5, max_iter=1000, penalty='l1',
                     solver='liblinear')

In [44]:
print(m3.score(X_train, y_train))
print(m3.score(X_test, y_test))

0.9946236559139785
0.9883512544802867


In [45]:
metrics.confusion_matrix(y_test, m3.predict(X_test))

array([[892,   6],
       [  7, 211]])

In [46]:
# We've got 7 Type II error in our results!
# The recall is 211 / 218 = 96.8%
# That is, we correctly detected 96.8% of all malicious software.

In [47]:
211/218

0.9678899082568807

### 8. Model Selection

Look at all three models you've produced. Which is best? Justify your answer.

In [48]:
# Both the l1 and l2-regularized models have very high predictive accuracy
# and very comparable recall scores. Either model would be acceptable.
# However, if one must be used, I think l1 is generally preferable due to
# its selection property.